In [1]:
!pip install pymupdf langchain langchain-text-splitters -q
from google.colab import drive
drive.mount('/content/drive')



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 69.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os, re, json
import fitz  # PyMuPDF

PDF_PATH   = "/content/Medical_Book.pdf"
DOC_NAME   = "encyclopedia"
BATCH_SIZE = 10

WORK_DIR   = "/content/drive/MyDrive/rag"
CHECKPOINT = f"{WORK_DIR}/{DOC_NAME}_checkpoint.json"
CHUNKS_OUT = f"{WORK_DIR}/{DOC_NAME}_chunks.jsonl"
IMAGES_DIR = f"{WORK_DIR}/{DOC_NAME}_images"

EMB_MODEL   = "BAAI/bge-small-en-v1.5"   # 384-dim; ganti sesuai kebutuhan
EMB_DIM     = 384                        # HARUS cocok dgn model di atas
# BGE query perlu prefix ini; passage (dokumen) TIDAK perlu prefix.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
EMB_BATCH   = 64
import uuid  # tambahkan di baris import paling atas config
EMB_CKPT = f"{WORK_DIR}/{DOC_NAME}_embed_checkpoint.json"  # tambahkan di blok path

# ---- Qdrant Cloud ----
QDRANT_URL     = "https://YOUR-CLUSTER.qdrant.io:6333"
QDRANT_API_KEY = "YOUR_QDRANT_API_KEY"
COLLECTION     = "encyclopedia"
UPSERT_BATCH   = 128


# --- parameter layout (bisa dikalibrasi lewat Cell 9) ---
FORCE_TWO_COLUMN   = True   # False -> auto-detect per halaman (lihat is_two_column)
HEADING_SIZE_RATIO = 1.15   # font >= median*ratio dianggap heading kandidat
H1_SIZE_RATIO      = 1.5    # font >= median*ratio dianggap H1 (judul entri besar)
MIN_GUTTER_FRAC    = 0.35   # gutter dicari di 35%-65% lebar halaman
MAX_GUTTER_FRAC    = 0.65

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMB_MODEL, device=device)
print(f"loaded {EMB_MODEL} on {device} | dim={model.get_sentence_embedding_dimension()}")
assert model.get_sentence_embedding_dimension() == EMB_DIM, \
    "EMB_DIM tidak cocok dengan model! Sesuaikan EMB_DIM."


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

if not client.collection_exists(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=EMB_DIM, distance=Distance.COSINE),
    )
    print(f"created collection '{COLLECTION}' (dim={EMB_DIM}, cosine)")
else:
    info = client.get_collection(COLLECTION)
    print(f"collection '{COLLECTION}' exists | points={info.points_count}")


In [ ]:
def load_chunks(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_embed_offset():
    if os.path.exists(EMB_CKPT):
        with open(EMB_CKPT) as f:
            return json.load(f).get("offset", 0)
    return 0

def save_embed_offset(offset):
    with open(EMB_CKPT, "w") as f:
        json.dump({"offset": offset, "doc": DOC_NAME}, f)

def stable_id(doc_name, idx):
    """ID deterministik supaya re-run meng-upsert titik yang sama (idempoten),
    bukan menduplikasi."""
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{doc_name}:{idx}"))


In [5]:
def save_image_bytes(img_bytes, ext, object_key_noext):
    """Simpan bytes gambar ke Drive. object_key mempertahankan struktur
    doc/header_slug/page/img_n supaya mudah dipindah ke B2 nanti.
    Return path lokal sebagai pointer metadata."""
    ext = (ext or "png").lower()
    if ext == "jpeg":
        ext = "jpg"
    dest = os.path.join(IMAGES_DIR, f"{object_key_noext}.{ext}")
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, "wb") as f:
        f.write(img_bytes)
    return dest

In [12]:
def slugify(text):
    text = (text or "").strip().lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "no_header"

def load_offset():
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT) as f:
            return json.load(f).get("offset", 0)
    return 0

def save_offset(offset):
    with open(CHECKPOINT, "w") as f:
        json.dump({"offset": offset, "doc": DOC_NAME}, f)

def append_chunks(records):
    with open(CHUNKS_OUT, "a") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")



In [13]:
def get_text_blocks(page):
    """Ambil blok teks dengan koordinat & info span (ukuran/berat font).
    Return list of dict: {bbox, lines:[{spans:[{text,size,flags,font}]}]}."""
    d = page.get_text("dict")
    blocks = []
    for b in d["blocks"]:
        if b.get("type") != 0:      # 0 = text block (1 = image)
            continue
        blocks.append(b)
    return blocks

def median(values):
    if not values:
        return 0
    s = sorted(values)
    n = len(s)
    return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2

def page_font_stats(blocks):
    """Kumpulkan semua ukuran font untuk menentukan median (baseline body)."""
    sizes = []
    for b in blocks:
        for line in b.get("lines", []):
            for span in line.get("spans", []):
                if span["text"].strip():
                    sizes.append(round(span["size"], 1))
    return median(sizes), sizes

def detect_gutter(page, blocks):
    """Cari koordinat-x pemisah kolom via celah kosong di tengah halaman.
    Bangun histogram cakupan-x dari bbox blok; cari x dgn cakupan minimum
    di rentang tengah (MIN_GUTTER_FRAC..MAX_GUTTER_FRAC)."""
    W = page.rect.width
    lo, hi = W * MIN_GUTTER_FRAC, W * MAX_GUTTER_FRAC
    bins = 100
    coverage = [0] * bins
    def xbin(x):
        return min(bins - 1, max(0, int(x / W * bins)))
    for b in blocks:
        x0, _, x1, _ = b["bbox"]
        for i in range(xbin(x0), xbin(x1) + 1):
            coverage[i] += 1
    # cari bin dengan coverage minimum di rentang tengah
    center_bins = [(coverage[i], i) for i in range(bins)
                   if lo <= (i / bins * W) <= hi]
    if not center_bins:
        return W / 2
    _, best_i = min(center_bins, key=lambda t: t[0])
    return best_i / bins * W

def is_two_column(page, blocks, gutter):
    """Heuristik: apakah ada cukup blok di kedua sisi gutter?"""
    left = sum(1 for b in blocks if (b["bbox"][0] + b["bbox"][2]) / 2 < gutter)
    right = len(blocks) - left
    return left >= 2 and right >= 2

def order_blocks_reading(blocks, gutter, two_col):
    """Urutkan blok sesuai reading order.
    2 kolom: kiri (sort by y) dulu, lalu kanan (sort by y).
    1 kolom: murni top->bottom."""
    if two_col:
        left  = [b for b in blocks if (b["bbox"][0] + b["bbox"][2]) / 2 <  gutter]
        right = [b for b in blocks if (b["bbox"][0] + b["bbox"][2]) / 2 >= gutter]
        left.sort(key=lambda b: b["bbox"][1])
        right.sort(key=lambda b: b["bbox"][1])
        return left + right
    else:
        return sorted(blocks, key=lambda b: b["bbox"][1])


In [14]:
def block_dominant_span(block):
    """Ambil ukuran font & teks representatif sebuah blok (span terbesar)."""
    best_size, texts = 0, []
    bold = False
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            t = span["text"]
            if t.strip():
                texts.append(t)
                if span["size"] > best_size:
                    best_size = span["size"]
                # flags bit 4 (16) = bold pada PyMuPDF
                bold = bold or bool(span["flags"] & 16)
    return best_size, bold, " ".join(texts).strip()

def block_to_markdown(block, body_median):
    """Konversi satu blok jadi baris Markdown. Heading -> #/##, else paragraf."""
    size, bold, text = block_dominant_span(block)
    if not text:
        return ""
    if body_median and size >= body_median * H1_SIZE_RATIO:
        return f"# {text}\n"
    if body_median and (size >= body_median * HEADING_SIZE_RATIO or
                        (bold and size >= body_median * 1.05)):
        return f"## {text}\n"
    return text + "\n"

def page_to_markdown(page):
    """Proses satu halaman -> (markdown, current_header_slug_tracker).
    Juga return mapping heading terakhir untuk penamaan gambar."""
    blocks = get_text_blocks(page)
    body_median, _ = page_font_stats(blocks)
    gutter = detect_gutter(page, blocks) if FORCE_TWO_COLUMN or True else page.rect.width / 2
    two_col = True if FORCE_TWO_COLUMN else is_two_column(page, blocks, gutter)

    ordered = order_blocks_reading(blocks, gutter, two_col)

    md_lines = []
    last_header = "no_header"
    for b in ordered:
        line = block_to_markdown(b, body_median)
        if line.startswith("#"):
            last_header = slugify(line.lstrip("#").strip())
        md_lines.append(line)
    return "\n".join(md_lines), last_header


In [15]:
def extract_page_images(doc, page, page_index, header_slug):
    """Ekstrak semua gambar di halaman, simpan ke Drive.
    Return list path lokal (pointer). Key: doc/header_slug/page/img_n."""
    urls = []
    for n, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        try:
            base = doc.extract_image(xref)     # {'image': bytes, 'ext': 'png'/'jpeg'...}
        except Exception:
            continue
        key_noext = f"{DOC_NAME}/{header_slug}/{page_index}/img_{n}"
        path = save_image_bytes(base["image"], base.get("ext", "png"), key_noext)
        urls.append(path)
    return urls


In [16]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

HEADERS_ON = [("#", "h1"), ("##", "h2")]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS_ON,
                                             strip_headers=False)
body_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)

def process_batch(doc, batch_start, total_pages):
    end = min(batch_start + BATCH_SIZE, total_pages)
    # kumpulkan markdown seluruh batch + petakan gambar per halaman
    batch_md_parts = []
    page_images = {}   # page_index -> list of image paths
    page_header = {}   # page_index -> header_slug terakhir di page itu

    for pidx in range(batch_start, end):
        page = doc[pidx]
        md, header_slug = page_to_markdown(page)
        page_header[pidx] = header_slug
        # sisipkan penanda halaman (komentar) agar bisa memetakan chunk->page
        batch_md_parts.append(f"<!-- PAGE:{pidx} -->\n{md}")
        page_images[pidx] = extract_page_images(doc, page, pidx, header_slug)

    batch_md = "\n\n".join(batch_md_parts)

    # split berdasarkan heading, lalu size-limit
    sections = header_splitter.split_text(batch_md)
    chunks = []
    for sec in sections:
        for piece in body_splitter.split_documents([sec]):
            chunks.append(piece)

    # OPTION 2: kaitkan gambar ke chunk berdasarkan penanda PAGE di dalam teks chunk
    PAGE_RE = re.compile(r"<!-- PAGE:(\d+) -->")
    records = []
    for chunk in chunks:
        pages_in_chunk = [int(x) for x in PAGE_RE.findall(chunk.page_content)]
        # bersihkan penanda dari teks final
        clean_text = PAGE_RE.sub("", chunk.page_content).strip()

        image_urls = []
        for p in pages_in_chunk:
            image_urls.extend(page_images.get(p, []))

        headers_meta = {k: chunk.metadata.get(k)
                        for k in ("h1", "h2") if chunk.metadata.get(k)}
        records.append({
            "text": clean_text,
            "metadata": {
                "doc": DOC_NAME,
                "headers": headers_meta,
                "pages": pages_in_chunk,
                "has_image": len(image_urls) > 0,
                "image_urls": image_urls,
            },
        })
    return records, end


def run():
    doc = fitz.open(PDF_PATH)
    total_pages = doc.page_count
    offset = load_offset()
    print(f"total pages={total_pages}, resuming from offset={offset}")
    while offset < total_pages:
        print(f"[batch] pages {offset}-{min(offset+BATCH_SIZE, total_pages)-1}")
        records, end = process_batch(doc, offset, total_pages)
        append_chunks(records)
        offset = end
        save_offset(offset)
        print(f"  -> {len(records)} chunks | next offset={offset}")
    doc.close()
    print(f"DONE. chunks -> {CHUNKS_OUT}")
run()   # <- uncomment untuk mulai/lanjut


total pages=637, resuming from offset=0
[batch] pages 0-9
  -> 22 chunks | next offset=10
[batch] pages 10-19
  -> 56 chunks | next offset=20
[batch] pages 20-29
  -> 66 chunks | next offset=30
[batch] pages 30-39
  -> 71 chunks | next offset=40
[batch] pages 40-49
  -> 71 chunks | next offset=50
[batch] pages 50-59
  -> 54 chunks | next offset=60
[batch] pages 60-69
  -> 74 chunks | next offset=70
[batch] pages 70-79
  -> 74 chunks | next offset=80
[batch] pages 80-89
  -> 62 chunks | next offset=90
[batch] pages 90-99
  -> 52 chunks | next offset=100
[batch] pages 100-109
  -> 69 chunks | next offset=110
[batch] pages 110-119
  -> 74 chunks | next offset=120
[batch] pages 120-129
  -> 62 chunks | next offset=130
[batch] pages 130-139
  -> 52 chunks | next offset=140
[batch] pages 140-149
  -> 70 chunks | next offset=150
[batch] pages 150-159
  -> 62 chunks | next offset=160
[batch] pages 160-169
  -> 59 chunks | next offset=170
[batch] pages 170-179
  -> 64 chunks | next offset=180
[

In [ ]:
# === EMBED + UPSERT (jalankan SETELAH run() selesai) ===
def embed_passages(texts):
    return model.encode(texts, batch_size=EMB_BATCH,
                        normalize_embeddings=True,
                        show_progress_bar=False, convert_to_numpy=True)

def run_embed():
    chunks = load_chunks(CHUNKS_OUT)          # NOTE: definisikan CHUNKS_IN = CHUNKS_OUT
    total = len(chunks)
    offset = load_embed_offset()
    print(f"total chunks={total}, resuming from offset={offset}")
    while offset < total:
        batch = chunks[offset: offset + UPSERT_BATCH]
        vectors = embed_passages([c["text"] for c in batch])
        points = []
        for i, (c, vec) in enumerate(zip(batch, vectors)):
            payload = {
                "text": c["text"],
                "doc": c["metadata"].get("doc", DOC_NAME),
                "headers": c["metadata"].get("headers", {}),
                "pages": c["metadata"].get("pages", []),
                "has_image": c["metadata"].get("has_image", False),
                "image_urls": c["metadata"].get("image_urls", []),
            }
            points.append(PointStruct(id=stable_id(DOC_NAME, offset + i),
                                      vector=vec.tolist(), payload=payload))
        client.upsert(collection_name=COLLECTION, points=points, wait=True)
        offset += len(batch)
        save_embed_offset(offset)
        print(f"  upserted {offset}/{total}")
    print("DONE.")

run_embed()